In [3]:
import numpy as np
import pandas as pd
from yaml import safe_load
import os
from tqdm import tqdm

In [4]:
filenames = []
for file in os.listdir('t20s'):
    filenames.append(os.path.join('t20s',file))

In [5]:
filenames[0:5]

['t20s\\1001349.yaml',
 't20s\\1001351.yaml',
 't20s\\1001353.yaml',
 't20s\\1004729.yaml',
 't20s\\1007655.yaml']

In [6]:
import pandas as pd
import json
import yaml
from tqdm import tqdm

def get_records(text):
    """Fastest possible extraction with minimal branching."""
    text = text.strip()
    if not text: return []
    
    # Priority 1: Standard JSON (Fastest)
    if text.startswith('{') and text.endswith('}'):
        try: return [json.loads(text)]
        except: pass
    
    # Priority 2: JSON List
    if text.startswith('[') and text.endswith(']'):
        try: return json.loads(text)
        except: pass

    # Priority 3: YAML / JSON Lines (Slowest)
    try:
        # safe_load_all handles both YAML docs and JSON lines effectively
        return [d for d in yaml.safe_load_all(text) if d is not None]
    except:
        return []

# 1. Use a plain list to store DICTIONARIES, not DataFrames
all_rows = []

print("Phase 1: Reading and Parsing Files...")
for counter, file in enumerate(tqdm(filenames)):
    try:
        with open(file, 'r', encoding='utf-8') as f:
            data = get_records(f.read())
            
        if data:
            # Flattening logic: if data is a list of lists, flatten it
            if isinstance(data[0], list):
                data = [item for sublist in data for item in sublist]
            
            # Inject match_id into the dictionary directly
            for record in data:
                if isinstance(record, dict):
                    record['match_id'] = counter
                    all_rows.append(record)
    except Exception:
        continue

# 2. The Critical Step: Single conversion at the end
print(f"Phase 2: Converting {len(all_rows)} records to DataFrame...")

# 
# json_normalize is heavy. If your data isn't deeply nested, use pd.DataFrame()
if all_rows:
    # Try standard DataFrame first (10x faster than normalize)
    # Only use json_normalize if you have nested dictionaries like {"info": {"city": "Dubai"}}
    final_df = pd.json_normalize(all_rows) 
    print("Success!")
else:
    final_df = pd.DataFrame()

Phase 1: Reading and Parsing Files...


100%|██████████| 4802/4802 [16:52<00:00,  4.74it/s]


Phase 2: Converting 4801 records to DataFrame...
Success!


In [7]:
backup = final_df.copy()

In [8]:
final_df

,innings,match_id,meta.data_version,meta.created,meta.revision,info.balls_per_over,info.dates,info.gender,info.match_type,info.outcome.by.wickets,...,info.registry.people.Muhammad Kaleem,info.registry.people.Usman Mushtaq,info.registry.people.Mohammad Shahid,info.registry.people.Muktar Ali,info.registry.people.Mohammad Usman,info.registry.people.VS Wategaonkar,info.registry.people.Farhan Ahmed,info.registry.people.Khurram Manzoor,info.registry.people.P Negi,info.registry.people.Hafiz Qaleem
0,"[{'1st innings': {'team': 'Australia', 'delive...",0,0.91,2017-02-18,2,6,[2017-02-17],male,T20,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"[{'1st innings': {'team': 'Australia', 'delive...",1,0.91,2017-02-19,2,6,[2017-02-19],male,T20,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[{'1st innings': {'team': 'Australia', 'delive...",2,0.91,2017-02-23,1,6,[2017-02-22],male,T20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[{'1st innings': {'team': 'Hong Kong', 'delive...",3,0.91,2016-09-12,1,6,[2016-09-05],male,T20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"[{'1st innings': {'team': 'Zimbabwe', 'deliver...",4,0.91,2016-06-19,1,6,[2016-06-18],male,T20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",4796,0.91,2016-03-05,2,6,[2016-03-04],male,T20,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4797,"[{'1st innings': {'team': 'Bangladesh', 'deliv...",4797,0.91,2016-03-08,1,6,[2016-03-06],male,T20,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4798,"[{'1st innings': {'team': 'Netherlands', 'deli...",4798,0.91,2016-02-03,1,6,[2016-02-03],male,T20,NaN,...,NaN,NaN,NaN,NaN,cee89f44,NaN,927694f7,NaN,NaN,f566cd7d
4799,"[{'1st innings': {'team': 'Australia', 'delive...",4799,0.91,2016-09-12,1,6,[2016-09-06],male,T20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# 1. Define specific administrative columns to drop
cols_to_drop = [
    'meta.data_version',
    'meta.created',
    'meta.revision',
    'info.outcome.bowl_out',
    'info.bowl_out',
    'info.supersubs.South Africa',
    'info.supersubs.New Zealand',
    'info.outcome.eliminator',
    'info.outcome.result',
    'info.outcome.method',
    'info.neutral_venue',
    'info.match_type_number',
    'info.outcome.by.runs',
    'info.outcome.by.wickets'
]

# 2. Automatically find all 'registry' and 'players' columns (these are mostly NaN)
registry_cols = [col for col in final_df.columns if 'info.registry.people' in col]
player_list_cols = [col for col in final_df.columns if 'info.players' in col]

# 3. Combine and drop everything in one command
final_df.drop(columns=cols_to_drop + registry_cols + player_list_cols, inplace=True, errors='ignore')

# 4. Optional: View the clean columns remaining
print(final_df.columns)

Index(['innings', 'match_id', 'info.balls_per_over', 'info.dates',
       'info.gender', 'info.match_type', 'info.outcome.winner', 'info.overs',
       'info.player_of_match', 'info.teams', 'info.toss.decision',
       'info.toss.winner', 'info.umpires', 'info.venue', 'info.city',
       'info.toss.uncontested'],
      dtype='object')


In [10]:
final_df

,innings,match_id,info.balls_per_over,info.dates,info.gender,info.match_type,info.outcome.winner,info.overs,info.player_of_match,info.teams,info.toss.decision,info.toss.winner,info.umpires,info.venue,info.city,info.toss.uncontested
0,"[{'1st innings': {'team': 'Australia', 'delive...",0,6,[2017-02-17],male,T20,Sri Lanka,20,[DAS Gunaratne],"[Australia, Sri Lanka]",field,Sri Lanka,"[MD Martell, P Wilson]",Melbourne Cricket Ground,NaN,NaN
1,"[{'1st innings': {'team': 'Australia', 'delive...",1,6,[2017-02-19],male,T20,Sri Lanka,20,[DAS Gunaratne],"[Australia, Sri Lanka]",field,Sri Lanka,"[SD Fry, SJ Nogajski]","Simonds Stadium, South Geelong",Victoria,NaN
2,"[{'1st innings': {'team': 'Australia', 'delive...",2,6,[2017-02-22],male,T20,Australia,20,[A Zampa],"[Australia, Sri Lanka]",field,Sri Lanka,"[MD Martell, P Wilson]",Adelaide Oval,NaN,NaN
3,"[{'1st innings': {'team': 'Hong Kong', 'delive...",3,6,[2016-09-05],male,T20,Hong Kong,20,NaN,"[Ireland, Hong Kong]",bat,Hong Kong,"[R Black, AJ Neill]","Bready Cricket Club, Magheramason",Londonderry,NaN
4,"[{'1st innings': {'team': 'Zimbabwe', 'deliver...",4,6,[2016-06-18],male,T20,Zimbabwe,20,[E Chigumbura],"[Zimbabwe, India]",field,India,"[TJ Matibiri, RB Tiffin]",Harare Sports Club,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",4796,6,[2016-03-04],male,T20,Pakistan,20,[Umar Akmal],"[Pakistan, Sri Lanka]",field,Pakistan,"[AK Chaudhary, Enamul Haque]",Shere Bangla National Stadium,Mirpur,NaN
4797,"[{'1st innings': {'team': 'Bangladesh', 'deliv...",4797,6,[2016-03-06],male,T20,India,20,[S Dhawan],"[Bangladesh, India]",field,India,"[RSA Palliyaguruge, Shozab Raza]",Shere Bangla National Stadium,Mirpur,NaN
4798,"[{'1st innings': {'team': 'Netherlands', 'deli...",4798,6,[2016-02-03],male,T20,Netherlands,20,[Mudassar Bukhari],"[United Arab Emirates, Netherlands]",field,United Arab Emirates,"[CK Nandan, Sarika Prasad]",ICC Academy,Dubai,NaN
4799,"[{'1st innings': {'team': 'Australia', 'delive...",4799,6,[2016-09-06],male,T20,Australia,20,[GJ Maxwell],"[Sri Lanka, Australia]",field,Sri Lanka,"[REJ Martinesz, RR Wimalasiri]",Pallekele International Cricket Stadium,NaN,NaN


In [11]:

final_df.drop(columns=[
    'info.match_type',        
    'info.outcome.winner',    
    'info.player_of_match',   
    'info.umpires',           
    'info.toss.uncontested',  
    'info.balls_per_over'
],inplace=True)

In [12]:
final_df

,innings,match_id,info.dates,info.gender,info.overs,info.teams,info.toss.decision,info.toss.winner,info.venue,info.city
0,"[{'1st innings': {'team': 'Australia', 'delive...",0,[2017-02-17],male,20,"[Australia, Sri Lanka]",field,Sri Lanka,Melbourne Cricket Ground,NaN
1,"[{'1st innings': {'team': 'Australia', 'delive...",1,[2017-02-19],male,20,"[Australia, Sri Lanka]",field,Sri Lanka,"Simonds Stadium, South Geelong",Victoria
2,"[{'1st innings': {'team': 'Australia', 'delive...",2,[2017-02-22],male,20,"[Australia, Sri Lanka]",field,Sri Lanka,Adelaide Oval,NaN
3,"[{'1st innings': {'team': 'Hong Kong', 'delive...",3,[2016-09-05],male,20,"[Ireland, Hong Kong]",bat,Hong Kong,"Bready Cricket Club, Magheramason",Londonderry
4,"[{'1st innings': {'team': 'Zimbabwe', 'deliver...",4,[2016-06-18],male,20,"[Zimbabwe, India]",field,India,Harare Sports Club,NaN
...,...,...,...,...,...,...,...,...,...,...
4796,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",4796,[2016-03-04],male,20,"[Pakistan, Sri Lanka]",field,Pakistan,Shere Bangla National Stadium,Mirpur
4797,"[{'1st innings': {'team': 'Bangladesh', 'deliv...",4797,[2016-03-06],male,20,"[Bangladesh, India]",field,India,Shere Bangla National Stadium,Mirpur
4798,"[{'1st innings': {'team': 'Netherlands', 'deli...",4798,[2016-02-03],male,20,"[United Arab Emirates, Netherlands]",field,United Arab Emirates,ICC Academy,Dubai
4799,"[{'1st innings': {'team': 'Australia', 'delive...",4799,[2016-09-06],male,20,"[Sri Lanka, Australia]",field,Sri Lanka,Pallekele International Cricket Stadium,NaN


In [13]:
final_df['info.overs'].value_counts()

info.overs
20    4786
50      15
Name: count, dtype: int64

In [14]:
final_df = final_df[final_df['info.overs'] == 20]
final_df.drop(columns=['info.overs'],inplace=True)
final_df

C:\Users\Patel Meet\AppData\Local\Temp\ipykernel_32524\845537911.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.drop(columns=['info.overs'],inplace=True)


,innings,match_id,info.dates,info.gender,info.teams,info.toss.decision,info.toss.winner,info.venue,info.city
0,"[{'1st innings': {'team': 'Australia', 'delive...",0,[2017-02-17],male,"[Australia, Sri Lanka]",field,Sri Lanka,Melbourne Cricket Ground,NaN
1,"[{'1st innings': {'team': 'Australia', 'delive...",1,[2017-02-19],male,"[Australia, Sri Lanka]",field,Sri Lanka,"Simonds Stadium, South Geelong",Victoria
2,"[{'1st innings': {'team': 'Australia', 'delive...",2,[2017-02-22],male,"[Australia, Sri Lanka]",field,Sri Lanka,Adelaide Oval,NaN
3,"[{'1st innings': {'team': 'Hong Kong', 'delive...",3,[2016-09-05],male,"[Ireland, Hong Kong]",bat,Hong Kong,"Bready Cricket Club, Magheramason",Londonderry
4,"[{'1st innings': {'team': 'Zimbabwe', 'deliver...",4,[2016-06-18],male,"[Zimbabwe, India]",field,India,Harare Sports Club,NaN
...,...,...,...,...,...,...,...,...,...
4796,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",4796,[2016-03-04],male,"[Pakistan, Sri Lanka]",field,Pakistan,Shere Bangla National Stadium,Mirpur
4797,"[{'1st innings': {'team': 'Bangladesh', 'deliv...",4797,[2016-03-06],male,"[Bangladesh, India]",field,India,Shere Bangla National Stadium,Mirpur
4798,"[{'1st innings': {'team': 'Netherlands', 'deli...",4798,[2016-02-03],male,"[United Arab Emirates, Netherlands]",field,United Arab Emirates,ICC Academy,Dubai
4799,"[{'1st innings': {'team': 'Australia', 'delive...",4799,[2016-09-06],male,"[Sri Lanka, Australia]",field,Sri Lanka,Pallekele International Cricket Stadium,NaN


In [15]:
final_df.isnull().sum()

innings                 0
match_id                0
info.dates              0
info.gender             0
info.teams              0
info.toss.decision      0
info.toss.winner        0
info.venue              0
info.city             178
dtype: int64

In [16]:
final_df['info.city'] = final_df['info.city'].fillna('Unknown')
print(final_df.isnull().sum())

innings               0
match_id              0
info.dates            0
info.gender           0
info.teams            0
info.toss.decision    0
info.toss.winner      0
info.venue            0
info.city             0
dtype: int64


C:\Users\Patel Meet\AppData\Local\Temp\ipykernel_32524\1222946994.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['info.city'] = final_df['info.city'].fillna('Unknown')


In [17]:
import pickle
pickle.dump(final_df,open('dataset_level1.pkl','wb'))

In [18]:
matches = pickle.load(open('dataset_level1.pkl','rb'))
matches.iloc[0]['innings'][0]['1st innings']['deliveries']

[{0.1: {'batsman': 'AJ Finch',
   'bowler': 'SL Malinga',
   'non_striker': 'M Klinger',
   'runs': {'batsman': 0, 'extras': 0, 'total': 0}}},
 {0.2: {'batsman': 'AJ Finch',
   'bowler': 'SL Malinga',
   'non_striker': 'M Klinger',
   'runs': {'batsman': 0, 'extras': 0, 'total': 0}}},
 {0.3: {'batsman': 'AJ Finch',
   'bowler': 'SL Malinga',
   'non_striker': 'M Klinger',
   'runs': {'batsman': 1, 'extras': 0, 'total': 1}}},
 {0.4: {'batsman': 'M Klinger',
   'bowler': 'SL Malinga',
   'non_striker': 'AJ Finch',
   'runs': {'batsman': 2, 'extras': 0, 'total': 2}}},
 {0.5: {'batsman': 'M Klinger',
   'bowler': 'SL Malinga',
   'non_striker': 'AJ Finch',
   'runs': {'batsman': 0, 'extras': 0, 'total': 0}}},
 {0.6: {'batsman': 'M Klinger',
   'bowler': 'SL Malinga',
   'non_striker': 'AJ Finch',
   'runs': {'batsman': 3, 'extras': 0, 'total': 3}}},
 {1.1: {'batsman': 'M Klinger',
   'bowler': 'KMDN Kulasekara',
   'non_striker': 'AJ Finch',
   'runs': {'batsman': 0, 'extras': 0, 'total': 

In [19]:
count = 1
delivery_df = pd.DataFrame()
for index, row in matches.iterrows():
    if count in [75,108,150,180,268,360,443,458,584,748,982,1052,1111,1226,1345]:
        count+=1
        continue
    count+=1
    ball_of_match = []
    batsman = []
    bowler = []
    runs = []
    player_of_dismissed = []
    teams = []
    batting_team = []
    match_id = []
    city = []
    venue = []
    for ball in row['innings'][0]['1st innings']['deliveries']:
        for key in ball.keys():
            match_id.append(count)
            batting_team.append(row['innings'][0]['1st innings']['team'])
            teams.append(row['info.teams'])
            ball_of_match.append(key)
            batsman.append(ball[key]['batsman'])
            bowler.append(ball[key]['bowler'])
            runs.append(ball[key]['runs']['total'])
            city.append(row['info.city'])
            venue.append(row['info.venue'])
            try:
                player_of_dismissed.append(ball[key]['wicket']['player_out'])
            except:
                player_of_dismissed.append('0')
    loop_df = pd.DataFrame({
            'match_id':match_id,
            'teams':teams,
            'batting_team':batting_team,
            'ball':ball_of_match,
            'batsman':batsman,
            'bowler':bowler,
            'runs':runs,
            'player_dismissed':player_of_dismissed,
            'city':city,
            'venue':venue
        })
    delivery_df = pd.concat([delivery_df, loop_df], ignore_index=True)

In [20]:
delivery_df

,match_id,teams,batting_team,ball,batsman,bowler,runs,player_dismissed,city,venue
0,2,"[Australia, Sri Lanka]",Australia,0.1,AJ Finch,SL Malinga,0,0,Unknown,Melbourne Cricket Ground
1,2,"[Australia, Sri Lanka]",Australia,0.2,AJ Finch,SL Malinga,0,0,Unknown,Melbourne Cricket Ground
2,2,"[Australia, Sri Lanka]",Australia,0.3,AJ Finch,SL Malinga,1,0,Unknown,Melbourne Cricket Ground
3,2,"[Australia, Sri Lanka]",Australia,0.4,M Klinger,SL Malinga,2,0,Unknown,Melbourne Cricket Ground
4,2,"[Australia, Sri Lanka]",Australia,0.5,M Klinger,SL Malinga,0,0,Unknown,Melbourne Cricket Ground
...,...,...,...,...,...,...,...,...,...,...
581767,4787,"[Sri Lanka, Australia]",Sri Lanka,19.3,SMSM Senanayake,MA Starc,1,0,Colombo,R Premadasa Stadium
581768,4787,"[Sri Lanka, Australia]",Sri Lanka,19.4,DM de Silva,MA Starc,0,0,Colombo,R Premadasa Stadium
581769,4787,"[Sri Lanka, Australia]",Sri Lanka,19.5,DM de Silva,MA Starc,0,DM de Silva,Colombo,R Premadasa Stadium
581770,4787,"[Sri Lanka, Australia]",Sri Lanka,19.6,SMSM Senanayake,MA Starc,2,0,Colombo,R Premadasa Stadium


In [21]:
def bowl(row):
    for team in row['teams']:
        if team != row['batting_team']:
            return team

In [22]:
delivery_df['bowling_team'] = delivery_df.apply(bowl,axis=1)

In [23]:
delivery_df

,match_id,teams,batting_team,ball,batsman,bowler,runs,player_dismissed,city,venue,bowling_team
0,2,"[Australia, Sri Lanka]",Australia,0.1,AJ Finch,SL Malinga,0,0,Unknown,Melbourne Cricket Ground,Sri Lanka
1,2,"[Australia, Sri Lanka]",Australia,0.2,AJ Finch,SL Malinga,0,0,Unknown,Melbourne Cricket Ground,Sri Lanka
2,2,"[Australia, Sri Lanka]",Australia,0.3,AJ Finch,SL Malinga,1,0,Unknown,Melbourne Cricket Ground,Sri Lanka
3,2,"[Australia, Sri Lanka]",Australia,0.4,M Klinger,SL Malinga,2,0,Unknown,Melbourne Cricket Ground,Sri Lanka
4,2,"[Australia, Sri Lanka]",Australia,0.5,M Klinger,SL Malinga,0,0,Unknown,Melbourne Cricket Ground,Sri Lanka
...,...,...,...,...,...,...,...,...,...,...,...
581767,4787,"[Sri Lanka, Australia]",Sri Lanka,19.3,SMSM Senanayake,MA Starc,1,0,Colombo,R Premadasa Stadium,Australia
581768,4787,"[Sri Lanka, Australia]",Sri Lanka,19.4,DM de Silva,MA Starc,0,0,Colombo,R Premadasa Stadium,Australia
581769,4787,"[Sri Lanka, Australia]",Sri Lanka,19.5,DM de Silva,MA Starc,0,DM de Silva,Colombo,R Premadasa Stadium,Australia
581770,4787,"[Sri Lanka, Australia]",Sri Lanka,19.6,SMSM Senanayake,MA Starc,2,0,Colombo,R Premadasa Stadium,Australia


In [24]:
delivery_df.drop(columns=['teams'],inplace=True)

In [25]:
delivery_df

,match_id,batting_team,ball,batsman,bowler,runs,player_dismissed,city,venue,bowling_team
0,2,Australia,0.1,AJ Finch,SL Malinga,0,0,Unknown,Melbourne Cricket Ground,Sri Lanka
1,2,Australia,0.2,AJ Finch,SL Malinga,0,0,Unknown,Melbourne Cricket Ground,Sri Lanka
2,2,Australia,0.3,AJ Finch,SL Malinga,1,0,Unknown,Melbourne Cricket Ground,Sri Lanka
3,2,Australia,0.4,M Klinger,SL Malinga,2,0,Unknown,Melbourne Cricket Ground,Sri Lanka
4,2,Australia,0.5,M Klinger,SL Malinga,0,0,Unknown,Melbourne Cricket Ground,Sri Lanka
...,...,...,...,...,...,...,...,...,...,...
581767,4787,Sri Lanka,19.3,SMSM Senanayake,MA Starc,1,0,Colombo,R Premadasa Stadium,Australia
581768,4787,Sri Lanka,19.4,DM de Silva,MA Starc,0,0,Colombo,R Premadasa Stadium,Australia
581769,4787,Sri Lanka,19.5,DM de Silva,MA Starc,0,DM de Silva,Colombo,R Premadasa Stadium,Australia
581770,4787,Sri Lanka,19.6,SMSM Senanayake,MA Starc,2,0,Colombo,R Premadasa Stadium,Australia


In [26]:
delivery_df['batting_team'].unique()

array(['Australia', 'Hong Kong', 'Zimbabwe', 'India', 'Bangladesh',
       'New Zealand', 'South Africa', 'England', 'West Indies',
       'Sri Lanka', 'Pakistan', 'Scotland', 'Oman', 'Ireland',
       'Papua New Guinea', 'United Arab Emirates', 'Netherlands',
       'Thailand', 'Uganda', 'Malaysia', 'Botswana', 'Malawi',
       'Sierra Leone', 'Mozambique', 'Namibia', 'Lesotho', 'Nepal',
       'China', 'Kuwait', 'Vanuatu', 'Philippines',
       'United States of America', 'Germany', 'Nigeria', 'Tanzania',
       'Japan', 'Indonesia', 'Fiji', 'Samoa', 'Ghana', 'Kenya',
       'Guernsey', 'Denmark', 'Jersey', 'Italy', 'Norway', 'Maldives',
       'Mali', 'Singapore', 'Bermuda', 'Canada', 'Cayman Islands',
       'Portugal', 'Gibraltar', 'Spain', 'Bhutan', 'Qatar', 'Iran',
       'Austria', 'Belgium', 'Isle of Man', 'Bulgaria', 'Romania',
       'Luxembourg', 'Czech Republic', 'Rwanda', 'Greece', 'Serbia',
       'Malta', 'France', 'Sweden', 'Finland', 'Eswatini', 'Cameroon',
       'Hu

In [27]:
teams = [
    'India', 'Australia', 'England', 'South Africa', 'Pakistan', 
    'New Zealand', 'West Indies', 'Sri Lanka', 'Bangladesh', 
    'Afghanistan', 'Ireland', 'Zimbabwe', 'Netherlands', 
    'Scotland', 'Nepal', 'Namibia',  
    'United States of America', 'Nepal', 'Canada'
]

In [28]:

delivery_df = delivery_df[delivery_df['batting_team'].isin(teams)]
delivery_df = delivery_df[delivery_df['bowling_team'].isin(teams)]

In [29]:

delivery_df

,match_id,batting_team,ball,batsman,bowler,runs,player_dismissed,city,venue,bowling_team
0,2,Australia,0.1,AJ Finch,SL Malinga,0,0,Unknown,Melbourne Cricket Ground,Sri Lanka
1,2,Australia,0.2,AJ Finch,SL Malinga,0,0,Unknown,Melbourne Cricket Ground,Sri Lanka
2,2,Australia,0.3,AJ Finch,SL Malinga,1,0,Unknown,Melbourne Cricket Ground,Sri Lanka
3,2,Australia,0.4,M Klinger,SL Malinga,2,0,Unknown,Melbourne Cricket Ground,Sri Lanka
4,2,Australia,0.5,M Klinger,SL Malinga,0,0,Unknown,Melbourne Cricket Ground,Sri Lanka
...,...,...,...,...,...,...,...,...,...,...
581767,4787,Sri Lanka,19.3,SMSM Senanayake,MA Starc,1,0,Colombo,R Premadasa Stadium,Australia
581768,4787,Sri Lanka,19.4,DM de Silva,MA Starc,0,0,Colombo,R Premadasa Stadium,Australia
581769,4787,Sri Lanka,19.5,DM de Silva,MA Starc,0,DM de Silva,Colombo,R Premadasa Stadium,Australia
581770,4787,Sri Lanka,19.6,SMSM Senanayake,MA Starc,2,0,Colombo,R Premadasa Stadium,Australia


In [30]:
output = delivery_df[['match_id','batting_team','bowling_team','ball','runs','player_dismissed','city','venue']]

In [31]:
output

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,venue
0,2,Australia,Sri Lanka,0.1,0,0,Unknown,Melbourne Cricket Ground
1,2,Australia,Sri Lanka,0.2,0,0,Unknown,Melbourne Cricket Ground
2,2,Australia,Sri Lanka,0.3,1,0,Unknown,Melbourne Cricket Ground
3,2,Australia,Sri Lanka,0.4,2,0,Unknown,Melbourne Cricket Ground
4,2,Australia,Sri Lanka,0.5,0,0,Unknown,Melbourne Cricket Ground
...,...,...,...,...,...,...,...,...
581767,4787,Sri Lanka,Australia,19.3,1,0,Colombo,R Premadasa Stadium
581768,4787,Sri Lanka,Australia,19.4,0,0,Colombo,R Premadasa Stadium
581769,4787,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo,R Premadasa Stadium
581770,4787,Sri Lanka,Australia,19.6,2,0,Colombo,R Premadasa Stadium


In [32]:
pickle.dump(output,open('dataset_level2.pkl','wb'))